In [ ]:
import sys, os
from pathlib import Path
import torch
import torch.nn.functional as F
from torch.nn import CTCLoss, CrossEntropyLoss
from torch.utils.data import DataLoader
from itertools import chain
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')  # Headless backend
import matplotlib.pyplot as plt

PROJECT_PATH = Path(r"B:\College\DL\handwriting_autocomplete_system\phase3_style_transfer")
os.chdir(PROJECT_PATH)
sys.path.insert(0, str(PROJECT_PATH))

# Create output directory
OUTPUT_DIR = PROJECT_PATH / 'output_files'
OUTPUT_DIR.mkdir(exist_ok=True)

# Logging setup - redirect prints to file
class Logger:
    def __init__(self, filepath):
        self.terminal = sys.stdout
        self.log = open(filepath, 'w', buffering=1)
    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
    def flush(self):
        self.terminal.flush()
        self.log.flush()

sys.stdout = Logger(OUTPUT_DIR / 'training_log.txt')
epoch_log = open(OUTPUT_DIR / 'epochs.txt', 'w', buffering=1)

from lib.datasets import get_dataset, get_collect_fn
from lib.alphabet import strLabelConverter, get_lexicon, get_true_alphabet
from lib.utils import yaml2config
from networks.BigGAN_networks import Generator, Discriminator, PatchDiscriminator
from networks.module import Recognizer, WriterIdentifier, StyleEncoder, StyleBackbone
from networks.loss import recn_l1_loss, CXLoss, KLloss
from networks.rand_dist import prepare_z_dist, prepare_y_dist
from networks.utils import get_scheduler, idx_to_words, set_requires_grad, extract_all_patches

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Output directory: {OUTPUT_DIR}")

Load the data for train

In [ ]:
# Config
cfg = yaml2config(str(PROJECT_PATH / 'configs' / 'gan_iam.yml'))
cfg.device = str(device)
cfg.training.batch_size = 24
cfg.training.epochs = 70

# Dataset
collect_fn = get_collect_fn(cfg.training.sort_input, sort_style=True)
train_dataset = get_dataset(cfg.dataset, cfg.training.dset_split, recogn_aug=True, wid_aug=True, process_style=True)
train_loader = DataLoader(train_dataset, batch_size=cfg.training.batch_size, shuffle=True, collate_fn=collect_fn, num_workers=0, drop_last=True)
print(f"Training samples: {len(train_dataset)}, batches: {len(train_loader)}")

In [ ]:
# Models
generator = Generator(**cfg.GenModel).to(device)
discriminator = Discriminator(**cfg.DiscModel).to(device)
patch_discriminator = PatchDiscriminator(**cfg.PatchDiscModel).to(device)
style_backbone = StyleBackbone(**cfg.StyBackbone).to(device)
style_encoder = StyleEncoder(**cfg.EncModel).to(device)
recognizer = Recognizer(**cfg.OcrModel).to(device)
writer_identifier = WriterIdentifier(**cfg.WidModel).to(device)

# Load pretrained OCR & WID
if os.path.exists(cfg.training.pretrained_r):
    recognizer.load_state_dict(torch.load(cfg.training.pretrained_r, map_location=device)['Recognizer'])
if os.path.exists(cfg.training.pretrained_w):
    w_dict = torch.load(cfg.training.pretrained_w, map_location=device)
    writer_identifier.load_state_dict(w_dict['WriterIdentifier'])
    style_backbone.load_state_dict(w_dict['StyleBackbone'])

# Freeze auxiliary networks
for p in recognizer.parameters(): p.requires_grad = False
for p in writer_identifier.parameters(): p.requires_grad = False
print("Models initialized")

In [ ]:
# Optimizers
optimizer_G = torch.optim.Adam(chain(generator.parameters(), style_encoder.parameters()), lr=cfg.training.lr, betas=(cfg.training.adam_b1, cfg.training.adam_b2))
optimizer_D = torch.optim.Adam(chain(discriminator.parameters(), patch_discriminator.parameters()), lr=cfg.training.lr * 0.5, betas=(cfg.training.adam_b1, cfg.training.adam_b2))
lr_scheduler_G = get_scheduler(optimizer_G, cfg.training)
lr_scheduler_D = get_scheduler(optimizer_D, cfg.training)

# Loss functions
ctc_loss = CTCLoss(zero_infinity=True, reduction='mean')  #connectionist temporal classification
classify_loss = CrossEntropyLoss()
contextual_loss = CXLoss()

# Lexicon
lexicon = get_lexicon(cfg.training.lexicon, get_true_alphabet(cfg.dataset), max_length=cfg.training.max_word_len)
if not lexicon:
    alphabet = get_true_alphabet(cfg.dataset)
    lexicon = sorted(set(''.join(chr(c) for c in train_dataset.lbs[s:s+l] if chr(c) in alphabet).lower() 
                         for s, l in zip(train_dataset.lb_seek_idxs, train_dataset.lb_lens) if 1 < l < cfg.training.max_word_len))

# Random distributions
z_dist = prepare_z_dist(cfg.training.batch_size, cfg.EncModel.style_dim, device, seed=cfg.seed)
y_dist = prepare_y_dist(cfg.training.batch_size, len(lexicon), device, seed=cfg.seed)
label_converter = strLabelConverter('all')
print(f"Lexicon: {len(lexicon)} words")

In [ ]:
# =============================================================================
# TRAINING LOOP - HiGAN+ Handwriting Generation
# =============================================================================
# This implements adversarial training with multiple objectives:
# 1. Adversarial Loss: Fool discriminator into thinking generated images are real
# 2. CTC Loss: Ensure generated text is readable by OCR (content preservation)
# 3. Writer ID Loss: Preserve writer's style characteristics
# 4. Reconstruction Loss: When given same text, output should match input
# 5. Contextual Loss: Match feature distributions between real and generated
# 6. KL Loss (VAE mode): Regularize latent space to be Gaussian

vae_mode = cfg.training.vae_mode  # VAE mode adds KL divergence for smoother latent space
ctc_len_scale = recognizer.len_scale  # OCR network's downsampling factor (typically 4)
history = {'epoch': [], 'g_loss': [], 'd_loss': [], 'adv_loss': [], 'ctc_loss': [], 'wid_loss': [], 'recn_loss': []}
iter_count = 0
start_epoch = 1

# === Resume from checkpoint if available ===
RESUME = True
if RESUME:
    ckpt_dir = PROJECT_PATH / 'checkpoints'
    if ckpt_dir.exists():
        ckpts = sorted(ckpt_dir.glob('epoch_*.pth'), key=lambda x: int(x.stem.split('_')[1]))
        if ckpts:
            latest_ckpt = ckpts[-1]
            print(f"Resuming from {latest_ckpt.name}...")
            ckpt = torch.load(latest_ckpt, map_location=device)
            
            generator.load_state_dict(ckpt['generator'])
            style_encoder.load_state_dict(ckpt['style_encoder'])
            discriminator.load_state_dict(ckpt['discriminator'])
            patch_discriminator.load_state_dict(ckpt['patch_discriminator'])
            optimizer_G.load_state_dict(ckpt['optimizer_G'])
            optimizer_D.load_state_dict(ckpt['optimizer_D'])
            
            start_epoch = ckpt['epoch'] + 1
            iter_count = ckpt.get('iter_count', 0)
            history = ckpt.get('history', history)
            
            print(f"Resumed from epoch {ckpt['epoch']}, starting at epoch {start_epoch}")

def save_loss_plots(history, output_dir):
    """Save loss plots to file."""
    if not history['epoch']: return
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    plots = [('g_loss', 'Generator', 'blue'), ('d_loss', 'Discriminator', 'orange'), ('adv_loss', 'Adversarial', 'green'),
             ('ctc_loss', 'CTC (OCR)', 'red'), ('wid_loss', 'Writer ID', 'purple'), ('recn_loss', 'Reconstruction', 'brown')]
    for ax, (key, title, color) in zip(axes.ravel(), plots):
        if key in history and history[key]:
            ax.plot(history['epoch'], history[key], color=color, linewidth=2)
            ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title(title); ax.grid(True, alpha=0.3)
    plt.suptitle(f'Training Curves (Epoch {history["epoch"][-1]})', fontweight='bold')
    plt.tight_layout()
    plt.savefig(output_dir / 'loss_curves.png', dpi=150, bbox_inches='tight')
    plt.close()

for epoch in range(start_epoch, cfg.training.epochs + 1):
    generator.train(); discriminator.train(); patch_discriminator.train(); style_encoder.train()
    epoch_g, epoch_d, epoch_adv, epoch_ctc, epoch_wid, epoch_recn = 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{cfg.training.epochs}', file=sys.stdout)
    for batch in pbar:
        # =====================================================================
        # BATCH DATA EXTRACTION
        # =====================================================================
        # style_imgs: Real handwriting images to learn style from
        # aug_imgs: Augmented versions (rotation, noise) for robust discrimination
        # lbs: Character labels encoded as integers (ground truth text)
        # wids: Writer IDs for style consistency loss
        real_imgs = batch['style_imgs'].to(device)
        real_img_lens = batch['style_img_lens'].to(device)
        real_aug_imgs = batch['aug_imgs'].to(device)
        real_aug_img_lens = batch['aug_img_lens'].to(device)
        real_lbs = batch['lbs'].to(device)
        real_lb_lens = batch['lb_lens'].to(device)
        real_wids = batch['wids'].to(device)
        max_label_len = real_lbs.size(-1)

        # =====================================================================
        # DISCRIMINATOR TRAINING (Every iteration)
        # =====================================================================
        # Goal: Learn to distinguish real handwriting from generated
        # Uses Hinge Loss: max(0, 1-D(real)) + max(0, 1+D(fake))
        # This pushes D(real) > 1 and D(fake) < -1 for stable training
        optimizer_D.zero_grad()
        set_requires_grad([generator, style_encoder], False)  # Freeze G during D training
        set_requires_grad([discriminator, patch_discriminator], True)
        
        with torch.no_grad():
            # Generate 3 types of fake images for diverse discrimination:
            # 1. RANDOM STYLE: z ~ N(0,1), random words from lexicon
            #    Tests if D can detect "invented" styles
            y_dist.sample_()
            fake_words = idx_to_words(y_dist, lexicon, max_label_len, cfg.training.capitalize_ratio, cfg.training.blank_ratio)
            fake_lbs, fake_lb_lens = label_converter.encode(fake_words, max_label_len)
            fake_lbs, fake_lb_lens = fake_lbs.to(device), fake_lb_lens.to(device)
            z_dist.sample_()
            fake_imgs = generator(z_dist, fake_lbs, fake_lb_lens)
            
            # 2. STYLE TRANSFER: Encode real image → style vector → generate new text
            #    Tests if D can detect style being applied to different content
            enc_z = style_encoder(real_imgs, real_img_lens, style_backbone, vae_mode=vae_mode) if not vae_mode else style_encoder(real_imgs, real_img_lens, style_backbone, vae_mode=True)[0]
            style_imgs = generator(enc_z, fake_lbs, fake_lb_lens)
            
            # 3. RECONSTRUCTION: Encode real → generate with SAME text
            #    Tests if D can detect subtle differences from original
            recn_imgs = generator(enc_z, real_lbs, real_lb_lens)
            
            cat_fake = torch.cat([fake_imgs, style_imgs, recn_imgs], dim=0)
            cat_lb_lens = torch.cat([fake_lb_lens, fake_lb_lens, real_lb_lens], dim=0)

        # Two-level discrimination for multi-scale realism:
        # 1. Full-image discriminator: Checks global coherence, overall style
        # 2. Patch discriminator: Checks local stroke quality, character details
        fake_disc = discriminator(cat_fake.detach(), cat_lb_lens * cfg.char_width, cat_lb_lens)
        fake_patch = patch_discriminator(extract_all_patches(cat_fake, cat_lb_lens * cfg.char_width).detach())
        real_disc = discriminator(real_imgs, real_img_lens, real_lb_lens)
        real_disc_aug = discriminator(real_aug_imgs, real_aug_img_lens, real_lb_lens)
        real_patch = patch_discriminator(torch.cat([extract_all_patches(real_imgs, real_img_lens), extract_all_patches(real_aug_imgs, real_aug_img_lens)], dim=0))
        
        # Hinge loss: Encourages margin of 1 between real/fake scores
        # ReLU(1 + D(fake)) → penalize if D(fake) > -1 (should be very negative)
        # ReLU(1 - D(real)) → penalize if D(real) < 1 (should be very positive)
        d_loss = (F.relu(1 + fake_disc).mean() + F.relu(1 + fake_patch).mean() + 
                  (F.relu(1 - real_disc).mean() + F.relu(1 - real_disc_aug).mean()) / 2 + F.relu(1 - real_patch).mean())
        d_loss.backward()
        torch.nn.utils.clip_grad_norm_(discriminator.parameters(), 5.0)  # Prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(patch_discriminator.parameters(), 5.0)
        optimizer_D.step()
        epoch_d += d_loss.item()

        # =====================================================================
        # GENERATOR TRAINING (Every num_critic_train iterations)
        # =====================================================================
        # Train G less frequently than D for stability (typically 1:2 or 1:5 ratio)
        # This gives D time to provide meaningful gradients to G
        if iter_count % cfg.training.num_critic_train == 0:
            optimizer_G.zero_grad()
            set_requires_grad([discriminator, patch_discriminator], False)  # Freeze D during G training
            set_requires_grad([generator, style_encoder], True)
            
            # Generate fresh samples for G training (different from D training samples)
            y_dist.sample_()
            fake_words = idx_to_words(y_dist, lexicon, max_label_len, cfg.training.capitalize_ratio, cfg.training.blank_ratio, sort=True)
            fake_lbs, fake_lb_lens = label_converter.encode(fake_words, max_label_len)
            fake_lbs, fake_lb_lens = fake_lbs.to(device), fake_lb_lens.to(device)
            z_dist.sample_()
            fake_imgs = generator(z_dist, fake_lbs, fake_lb_lens)
            
            # Style encoding with optional VAE reparameterization
            # VAE mode: z = μ + σ * ε where ε ~ N(0,1) → enables smooth interpolation
            # Non-VAE: Direct encoding → more precise but less generalizable
            if vae_mode:
                (enc_z, mu, logvar), real_feats = style_encoder(real_imgs, real_img_lens, style_backbone, ret_feats=True, vae_mode=True)
            else:
                enc_z, real_feats = style_encoder(real_imgs, real_img_lens, style_backbone, ret_feats=True, vae_mode=False)
            
            style_imgs = generator(enc_z, fake_lbs, fake_lb_lens)
            recn_imgs = generator(enc_z, real_lbs, real_lb_lens)
            
            # -----------------------------------------------------------------
            # LOSS 1: ADVERSARIAL LOSS (λ = 1.0)
            # -----------------------------------------------------------------
            # Maximize D(G(z)) → Generator tries to fool discriminator
            # Negative sign because we minimize: -D(fake) pushes D(fake) higher
            cat_fake = torch.cat([fake_imgs, style_imgs, recn_imgs], dim=0)
            cat_lb_lens = torch.cat([fake_lb_lens, fake_lb_lens, real_lb_lens], dim=0)
            adv_loss = -discriminator(cat_fake, cat_lb_lens * cfg.char_width, cat_lb_lens).mean()
            adv_patch = -patch_discriminator(extract_all_patches(cat_fake, cat_lb_lens * cfg.char_width)).mean()
            
            # -----------------------------------------------------------------
            # LOSS 2: CTC LOSS (λ = 3.0) - Content Preservation
            # -----------------------------------------------------------------
            # Connectionist Temporal Classification: Ensures generated text is readable
            # Frozen OCR network acts as "text critic" - if it can't read it, penalize G
            # Applied to all 3 image types to ensure text legibility everywhere
            ctc_rand = ctc_loss(recognizer(fake_imgs, fake_lb_lens * cfg.char_width), fake_lbs, fake_lb_lens * cfg.char_width // ctc_len_scale, fake_lb_lens)
            ctc_style = ctc_loss(recognizer(style_imgs, fake_lb_lens * cfg.char_width), fake_lbs, fake_lb_lens * cfg.char_width // ctc_len_scale, fake_lb_lens)
            ctc_recn = ctc_loss(recognizer(recn_imgs, real_lb_lens * cfg.char_width), real_lbs, real_lb_lens * cfg.char_width // ctc_len_scale, real_lb_lens)
            ctc_total = ctc_rand + ctc_style + ctc_recn
            
            # -----------------------------------------------------------------
            # LOSS 3: INFO LOSS (λ = 1.5) - Style Cycle Consistency
            # -----------------------------------------------------------------
            # If we encode a generated image, we should recover the original z
            # This ensures the generator actually uses the style vector, not ignoring it
            # |Encoder(Generator(z)) - z| should be small
            info_loss = (style_encoder(fake_imgs, fake_lb_lens * cfg.char_width, style_backbone) - z_dist.detach()).abs().mean()
            
            # -----------------------------------------------------------------
            # LOSS 4: RECONSTRUCTION LOSS (λ = 5.0) - Pixel-level Fidelity
            # -----------------------------------------------------------------
            # When encoding a real image and regenerating with SAME text:
            # Output should match input pixel-by-pixel (L1 distance)
            # This is the strongest supervision signal for learning style encoding
            recn_loss = recn_l1_loss(recn_imgs, real_imgs, real_img_lens)
            
            # -----------------------------------------------------------------
            # LOSS 5: WRITER ID LOSS (λ = 1.5) - Style Preservation
            # -----------------------------------------------------------------
            # Frozen writer classifier checks if generated images have correct writer style
            # Cross-entropy loss: Generated should be classified as same writer as input
            cat_style = torch.cat([style_imgs, recn_imgs], dim=0)
            wid_logits, fake_feats = writer_identifier(cat_style, torch.cat([fake_lb_lens, real_lb_lens]) * cfg.char_width, style_backbone, ret_feats=True)
            wid_loss = classify_loss(wid_logits, real_wids.repeat(2))
            
            # -----------------------------------------------------------------
            # LOSS 6: CONTEXTUAL LOSS (λ_ctx) - Feature Distribution Matching
            # -----------------------------------------------------------------
            # Matches feature distributions between real and generated at multiple scales
            # Uses cosine similarity in feature space rather than pixel space
            # More flexible than L1/L2 - allows spatial rearrangement of features
            ctx_loss = sum(contextual_loss(r, f) for r, f in zip(real_feats, [ff.chunk(2)[0] for ff in fake_feats]))
            
            # -----------------------------------------------------------------
            # LOSS 7: KL DIVERGENCE (λ_kl) - Latent Space Regularization
            # -----------------------------------------------------------------
            # Only in VAE mode: Forces latent distribution to be close to N(0,1)
            # KL(q(z|x) || p(z)) where q is encoder output, p is standard normal
            # Enables smooth interpolation and random sampling from latent space
            kl_loss = KLloss(mu, logvar) if vae_mode else torch.tensor(0.0, device=device)
            
            # -----------------------------------------------------------------
            # TOTAL GENERATOR LOSS - Weighted combination
            # -----------------------------------------------------------------
            # Weights reflect importance: Reconstruction (5.0) > CTC (3.0) > WID/Info (1.5) > Adversarial (1.0)
            # Higher weight = more important constraint for good handwriting generation
            g_loss = (adv_loss + adv_patch) + 3.0 * ctc_total + 1.5 * info_loss + 1.5 * wid_loss + 5.0 * recn_loss + cfg.training.lambda_ctx * ctx_loss + cfg.training.lambda_kl * kl_loss
            g_loss.backward()
            torch.nn.utils.clip_grad_norm_(generator.parameters(), 5.0)
            torch.nn.utils.clip_grad_norm_(style_encoder.parameters(), 5.0)
            optimizer_G.step()
            
            epoch_g += g_loss.item()
            epoch_adv += (adv_loss.item() + adv_patch.item())
            epoch_ctc += ctc_total.item()
            epoch_wid += wid_loss.item()
            epoch_recn += recn_loss.item()
        
        iter_count += 1
        pbar.set_postfix({'D': f'{d_loss.item():.3f}', 'G': f'{g_loss.item():.3f}' if iter_count % cfg.training.num_critic_train == 0 else '-'})
    
    # Epoch statistics - average losses over batches
    n_g = max(1, len(train_loader) // cfg.training.num_critic_train)
    history['epoch'].append(epoch)
    history['g_loss'].append(epoch_g / n_g)
    history['d_loss'].append(epoch_d / len(train_loader))
    history['adv_loss'].append(epoch_adv / n_g)
    history['ctc_loss'].append(epoch_ctc / n_g)
    history['wid_loss'].append(epoch_wid / n_g)
    history['recn_loss'].append(epoch_recn / n_g)
    
    epoch_msg = f"Epoch {epoch}/{cfg.training.epochs}: G={history['g_loss'][-1]:.4f}, D={history['d_loss'][-1]:.4f}, ADV={history['adv_loss'][-1]:.4f}, CTC={history['ctc_loss'][-1]:.4f}, WID={history['wid_loss'][-1]:.4f}, RECN={history['recn_loss'][-1]:.4f}"
    print(epoch_msg)
    epoch_log.write(epoch_msg + '\n')
    epoch_log.flush()
    
    os.makedirs('checkpoints', exist_ok=True)
    torch.save({
        'epoch': epoch, 'iter_count': iter_count, 'history': history,
        'generator': generator.state_dict(), 'style_encoder': style_encoder.state_dict(),
        'discriminator': discriminator.state_dict(), 'patch_discriminator': patch_discriminator.state_dict(),
        'optimizer_G': optimizer_G.state_dict(), 'optimizer_D': optimizer_D.state_dict(),
    }, f'checkpoints/epoch_{epoch}.pth')
    
    if epoch % 5 == 0 or epoch == cfg.training.epochs:
        save_loss_plots(history, OUTPUT_DIR)
        import csv
        with open(OUTPUT_DIR / 'training_history.csv', 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=history.keys())
            writer.writeheader()
            for i in range(len(history['epoch'])):
                writer.writerow({k: v[i] for k, v in history.items()})
    
    # Learning rate decay - typically step or cosine schedule
    # Gradually reduces LR for fine-tuning in later epochs
    lr_scheduler_G.step(epoch)
    lr_scheduler_D.step(epoch)

save_loss_plots(history, OUTPUT_DIR)
epoch_log.close()
print(f"\nTraining complete! Outputs saved to {OUTPUT_DIR}")